# PneumoScan --- ResNet-50 Transfer Learning

This notebook trains and evaluates **ResNet-50** using transfer learning for the PneumoScan multi-class chest X-ray classification task (Normal / Bacteria / Virus).

### What is Transfer Learning?

Transfer learning leverages a model pretrained on a large dataset (ImageNet --- 1.2 million images, 1000 classes) and adapts it to a new, smaller dataset. The pretrained convolutional layers have already learned rich, general-purpose visual features (edges, textures, shapes) that transfer well to medical imaging tasks. This approach:

- **Reduces training time** --- we only train the classification head from scratch
- **Improves generalization** --- pretrained features act as a strong regularizer
- **Works with smaller datasets** --- critical for medical imaging where labeled data is scarce

### ResNet-50 Architecture

ResNet-50 introduced **residual connections** (skip connections) that solve the vanishing gradient problem in deep networks:

```
x --> [Conv -> BN -> ReLU -> Conv -> BN] --> (+) --> ReLU
|                                           |
+-------------------------------------------+   (skip/identity connection)
```

Key properties:
- **50 layers deep** with ~23.5M parameters in the base
- Uses bottleneck blocks (1x1 -> 3x3 -> 1x1 convolutions) for efficiency
- Pretrained on ImageNet with 76.1% top-1 accuracy
- Skip connections enable training much deeper networks without degradation

### Two-Phase Training Strategy

We use a two-phase approach to prevent catastrophic forgetting of pretrained features:

| Phase | Layers | Learning Rate | Epochs | Purpose |
|-------|--------|---------------|--------|---------|
| **Phase 1: Feature Extraction** | Base frozen, head trainable | 1e-3 | 10 | Train the classification head to map ImageNet features to our 3 classes |
| **Phase 2: Fine-Tuning** | Top 30% of base unfrozen + head | 1e-5 | 20 | Gently adapt the higher convolutional layers to chest X-ray-specific features |

The low learning rate in Phase 2 prevents the pretrained weights from being overwritten too aggressively.

---
## 1. Environment Setup

In [ ]:
# Install dependencies (uncomment if running in Colab or fresh environment)
# !pip install -q tensorflow opencv-python matplotlib seaborn scikit-learn pandas

import sys
import os

# Add project root and src/ to Python path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
SRC_DIR = os.path.join(PROJECT_ROOT, "src")

for path in [PROJECT_ROOT, SRC_DIR]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source dir:   {SRC_DIR}")

In [ ]:
# Import project modules
import config
import data_loader
import models
import train
import utils
import evaluate

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Google Colab setup (no-op when running locally)
config.setup_colab()

# Create all required output directories
config.ensure_dirs()
print("Output directories ready.")

---
## 2. Load Data

Same pipeline as the baseline notebook: 80/20 stratified split with augmentation on the training portion.

In [ ]:
# Load training and validation datasets
train_ds, val_ds = data_loader.load_train_val_split()

print(f"Training batches:   {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")
print(f"Batch size:         {config.BATCH_SIZE}")
print(f"Image size:         {config.IMG_SIZE}")

---
## 3. Compute Class Weights

In [ ]:
class_weights = data_loader.compute_class_weights()
print(f"\nClass weights: {class_weights}")

---
## 4. Build the ResNet-50 Model

We load ResNet-50 with ImageNet weights and attach our custom classification head:

```
ResNet-50 base (frozen) --> GAP --> Dense(256) + BN + Dropout(0.3)
                                --> Dense(128) + BN + Dropout(0.3)
                                --> Dense(3, softmax)
```

Initially the base is **frozen** --- only the classification head is trainable.

In [ ]:
model = models.build_resnet50(freeze=True)
model.summary(show_trainable=True, expand_nested=False)

In [ ]:
# Count trainable vs frozen parameters
trainable_params = sum(np.prod(w.shape) for w in model.trainable_weights)
total_params = sum(np.prod(w.shape) for w in model.weights)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {frozen_params:>12,}")
print(f"Trainable fraction:   {trainable_params / total_params:.2%}")

---
## 5. Two-Phase Training

The `train.train_model()` function automatically handles both phases for transfer learning models:

**Phase 1 --- Feature Extraction (frozen base)**
- Only the classification head trains
- Learning rate: 1e-3 (can be aggressive since base weights are frozen)
- Goal: Learn to map ImageNet features to our 3-class problem

**Phase 2 --- Fine-Tuning (top 30% unfrozen)**
- Top 30% of ResNet-50 layers become trainable alongside the head
- Learning rate: 1e-5 (must be small to avoid catastrophic forgetting)
- Goal: Adapt high-level features (e.g., texture detectors) to chest X-ray patterns

Both phases use EarlyStopping, ReduceLROnPlateau, and ModelCheckpoint callbacks.

In [ ]:
model, history_phase1, history_phase2 = train.train_model(
    "resnet50",
    train_ds,
    val_ds,
    class_weights=class_weights,
)

---
## 6. Training Curves

Let's visualize both training phases to check for convergence, overfitting, and the effect of unfreezing.

In [ ]:
# Phase 1: Feature Extraction
print("Phase 1: Feature Extraction (frozen base)")
utils.plot_training_history(history_phase1, "ResNet-50 (Phase 1)")
plt.show()

In [ ]:
# Phase 2: Fine-Tuning
if history_phase2 is not None:
    print("Phase 2: Fine-Tuning (top 30% unfrozen)")
    utils.plot_training_history(history_phase2, "ResNet-50 (Phase 2)")
    plt.show()
else:
    print("Phase 2 was not executed.")

In [ ]:
# Print final metrics from both phases
print("Phase 1 Final Metrics:")
print(f"  Train Accuracy: {history_phase1.history['accuracy'][-1]:.4f}")
print(f"  Val Accuracy:   {history_phase1.history['val_accuracy'][-1]:.4f}")
print(f"  Train Loss:     {history_phase1.history['loss'][-1]:.4f}")
print(f"  Val Loss:       {history_phase1.history['val_loss'][-1]:.4f}")

if history_phase2 is not None:
    print("\nPhase 2 Final Metrics:")
    print(f"  Train Accuracy: {history_phase2.history['accuracy'][-1]:.4f}")
    print(f"  Val Accuracy:   {history_phase2.history['val_accuracy'][-1]:.4f}")
    print(f"  Train Loss:     {history_phase2.history['loss'][-1]:.4f}")
    print(f"  Val Loss:       {history_phase2.history['val_loss'][-1]:.4f}")

---
## 7. Evaluate on Test Set

Final evaluation on the held-out test set. We compute all metrics and generate confusion matrices, ROC curves, and precision-recall curves.

In [ ]:
# Load the test dataset
test_ds = data_loader.load_test_dataset()
print(f"Test batches: {len(test_ds)}")

In [ ]:
# Run full evaluation
metrics = evaluate.evaluate_model(model, test_ds, "resnet50")

print(f"\n{'=' * 40}")
print("ResNet-50 Test Results")
print(f"{'=' * 40}")
for key in ["accuracy", "f1_macro", "f1_weighted", "auc_roc", "auc_pr", "cohens_kappa", "inference_ms"]:
    print(f"  {key:18s}: {metrics[key]:.4f}")

---
## 8. Per-Class AUC Analysis

Let's examine the per-class AUC scores to understand which classes benefit most from transfer learning.

In [ ]:
print("Per-Class AUC-ROC:")
for cls_name, auc_val in metrics["auc_per_class"].items():
    print(f"  {cls_name:10s}: {auc_val:.4f}")

print("\nPer-Class Average Precision (AUC-PR):")
for cls_name, ap_val in metrics["ap_per_class"].items():
    print(f"  {cls_name:10s}: {ap_val:.4f}")

---
## Summary

| Aspect | Details |
|--------|--------|
| Model | ResNet-50 (ImageNet pretrained) + custom head |
| Phase 1 | Feature extraction --- frozen base, lr=1e-3, 10 epochs max |
| Phase 2 | Fine-tuning --- top 30% unfrozen, lr=1e-5, 20 epochs max |
| Augmentation | Random flip, rotation, zoom, translation, brightness |
| Class weights | Applied to handle imbalanced classes |

### Key Observations

- **Transfer learning significantly outperforms** the baseline custom CNN by leveraging rich pretrained features from ImageNet.
- **Phase 2 fine-tuning** typically provides an additional accuracy boost by adapting high-level features to the chest X-ray domain.
- **Residual connections** in ResNet-50 enable stable training even with 50 layers --- the skip connections prevent vanishing gradients.
- The most common confusion is typically between **Bacteria and Virus** classes, as both are types of pneumonia with overlapping radiographic patterns.

### Comparison with Baseline

| Metric | Custom CNN (Baseline) | ResNet-50 (Transfer) |
|--------|-----------------------|---------------------|
| Accuracy | (see NB02) | (see results above) |
| F1 Macro | (see NB02) | (see results above) |
| AUC-ROC | (see NB02) | (see results above) |

**Next steps:** Train EfficientNet-B0, DenseNet-121, and MobileNetV2 using the same two-phase approach, then compare all five models and build an ensemble.